In [1]:
!pip -q install -U google-genai pytest

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 12.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [4]:
import google.auth
import google.genai
import pytest

print("Google Auth version:", google.auth.__version__)
print("Gemini library: Working")
print("Pytest library: Working")

Google Auth version: 2.62.0
Gemini library: Working
Pytest library: Working


In [5]:
import os
from getpass import getpass
from google import genai

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.5-flash"

print("Gemini client created successfully!")

Enter your Gemini API key: ··········
Gemini client created successfully!


In [6]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Explain Python unit testing in one simple sentence."
)

print(response.text)

Python unit testing is the practice of writing small, automated tests to verify that individual pieces of your code (like functions or methods) work correctly on their own.


In [7]:
def code_analyzer(source_code):
    prompt = f"""
You are a Python Code Analyzer Agent.

Analyze the source code below.
Explain:
1. What the code does
2. The functions and parameters
3. The expected output
4. Possible errors or edge cases

Keep the explanation simple and clear.

SOURCE CODE:
{source_code}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text


print("Code Analyzer Agent created successfully!")

Code Analyzer Agent created successfully!


In [9]:
MODEL_NAME = "gemini-3.5-flash-lite"

print("Model changed to:", MODEL_NAME)

Model changed to: gemini-3.5-flash-lite


In [10]:
analysis = code_analyzer(sample_code)

print("===== CODE ANALYSIS =====")
print(analysis)

===== CODE ANALYSIS =====
Here is the analysis of the provided Python source code:

### 1. What the code does
This code defines a simple utility function named `add` that performs basic addition. It takes two numbers (or objects that support the `+` operator), adds them together, and returns the resulting sum.

---

### 2. The functions and parameters
* **Function:** `add(a, b)`
  * **`a` (Parameter):** The first value to be added.
  * **`b` (Parameter):** The second value to be added to the first.

---

### 3. The expected output
The output depends entirely on the inputs passed to the function:
* If you pass integers or floats (e.g., `add(2, 3)`), it returns their numerical sum (`5`).
* If you pass strings (e.g., `add("Hello, ", "World!")`), it returns concatenated text (`"Hello, World!"`).
* If you pass lists (e.g., `add([1, 2], [3, 4])`), it returns a combined list (`[1, 2, 3, 4]`).

---

### 4. Possible errors or edge cases
* **TypeError:** Because Python uses dynamic typing, passi

In [11]:
def test_planner(source_code, analysis):
    prompt = f"""
You are a Software Test Planner Agent.

Read the source code and its analysis.
Create a simple test plan with:
1. Normal test cases
2. Boundary test cases
3. Invalid input test cases
4. Expected result for each test

Do not write Python test code yet.
Return a numbered list.

SOURCE CODE:
{source_code}

CODE ANALYSIS:
{analysis}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text


print("Test Planner Agent created successfully!")

Test Planner Agent created successfully!


In [12]:
test_plan = test_planner(sample_code, analysis)

print("===== TEST PLAN =====")
print(test_plan)

===== TEST PLAN =====
Here is the test plan based on the source code and analysis:

### 1. Normal Test Cases
1. **Test Case:** `add(2, 3)` (Positive Integers)
   * **Expected Result:** Returns `5`
2. **Test Case:** `add(-5, 5)` (Positive and Negative Integers)
   * **Expected Result:** Returns `0`
3. **Test Case:** `add(2.5, 3.1)` (Floating-point Numbers)
   * **Expected Result:** Returns `5.6`
4. **Test Case:** `add("Hello, ", "World!")` (Strings for concatenation)
   * **Expected Result:** Returns `"Hello, World!"`
5. **Test Case:** `add([1, 2], [3, 4])` (Lists for combination)
   * **Expected Result:** Returns `[1, 2, 3, 4]`

### 2. Boundary Test Cases
6. **Test Case:** `add(0, 0)` (Adding Zeros)
   * **Expected Result:** Returns `0`
7. **Test Case:** `add(1e308, 1e308)` (Extremely large numbers / near float limit)
   * **Expected Result:** Returns `inf` or the correct large float sum without unexpected crashes
8. **Test Case:** `add(0.1, 0.2)` (Floating-point precision boundary)
  

In [13]:
def test_generator(source_code, test_plan):
    prompt = f"""
You are a Python Unit Test Generator Agent.

Write executable pytest tests for the source code.
Import the function from user_code.

Requirements:
1. Cover the test plan.
2. Use clear test function names.
3. Use assert for expected results.
4. Use pytest.raises(TypeError) for expected errors.
5. For floating-point calculations, use pytest.approx().
6. Return Python code only, without Markdown fences.
7. Do not use file operations, shell commands,
   network access, eval, or exec.

SOURCE CODE:
{source_code}

TEST PLAN:
{test_plan}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    if not response.text:
        raise ValueError("Test Generator returned an empty response.")

    return response.text.strip()


print("Test Generator Agent created successfully!")

Test Generator Agent created successfully!


In [14]:
generated_tests = test_generator(sample_code, test_plan)

print("===== GENERATED PYTHON UNIT TESTS =====")
print(generated_tests)

===== GENERATED PYTHON UNIT TESTS =====
import pytest
from user_code import add

def test_add_positive_integers():
    assert add(2, 3) == 5

def test_add_positive_and_negative_integers():
    assert add(-5, 5) == 0

def test_add_floating_point_numbers():
    assert add(2.5, 3.1) == pytest.approx(5.6)

def test_add_strings():
    assert add("Hello, ", "World!") == "Hello, World!"

def test_add_lists():
    assert add([1, 2], [3, 4]) == [1, 2, 3, 4]

def test_add_zeros():
    assert add(0, 0) == 0

def test_add_extremely_large_numbers():
    assert add(1e308, 1e308) == float('inf')

def test_add_floating_point_precision():
    assert add(0.1, 0.2) == pytest.approx(0.3)

def test_add_empty_strings():
    assert add("", "") == ""

def test_add_empty_lists():
    assert add([], []) == []

def test_add_incompatible_types_string_and_int():
    with pytest.raises(TypeError):
        add("Age: ", 25)

def test_add_missing_second_argument():
    with pytest.raises(TypeError):
        add(5)

de

In [15]:
def test_reviewer(source_code, generated_tests):
    prompt = f"""
You are a Software Test Reviewer Agent.

Review the generated pytest code.

Check:
1. Are the expected results correct?
2. Are normal and edge cases covered?
3. Are exceptions tested correctly?
4. Are there syntax or logical mistakes?
5. What tests are missing?

Do not claim that you executed the tests.
Give a short review and list suggested improvements.

SOURCE CODE:
{source_code}

GENERATED TESTS:
{generated_tests}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    if not response.text:
        raise ValueError("Test Reviewer returned an empty response.")

    return response.text.strip()


print("Test Reviewer Agent created successfully!")

Test Reviewer Agent created successfully!


In [16]:
review = test_reviewer(sample_code, generated_tests)

print("===== TEST REVIEW =====")
print(review)

===== TEST REVIEW =====
### Short Review

The generated test suite for the `add` function is remarkably comprehensive, well-structured, and idiomatic. It correctly uses `pytest.approx` for floating-point comparisons, tests multiple data types (integers, floats, strings, lists), and properly covers edge cases such as empty containers, zero values, extremely large numbers (overflow), and type errors. There are no syntax or logical errors in the test code.

### Suggested Improvements

While the current test suite is high quality, the following improvements and additions can make it even more robust:

1. **Add Tuple Addition Test:**
   * Since Python's `+` operator also supports tuples, it would be good to test adding two tuples (e.g., `add((1, 2), (3, 4)) == (1, 2, 3, 4)`).

2. **Test Commutativity Property (Optional Property-Based Testing):**
   * Although not strictly necessary for such a simple function, verifying that `add(a, b) == add(b, a)` for various inputs ensures logical consist

In [17]:
print("Agent 1: Code Analyzer     - Ready")
print("Agent 2: Test Planner      - Ready")
print("Agent 3: Test Generator    - Ready")
print("Agent 4: Test Reviewer     - Ready")

Agent 1: Code Analyzer     - Ready
Agent 2: Test Planner      - Ready
Agent 3: Test Generator    - Ready
Agent 4: Test Reviewer     - Ready


In [18]:
def run_multi_agent_system(source_code):

    print("1. Code Analyzer Agent running...")
    analysis = code_analyzer(source_code)

    print("2. Test Planner Agent running...")
    plan = test_planner(source_code, analysis)

    print("3. Test Generator Agent running...")
    tests = test_generator(source_code, plan)

    print("4. Test Reviewer Agent running...")
    review = test_reviewer(source_code, tests)

    print("\nAll four agents completed!")

    return {
        "source_code": source_code,
        "analysis": analysis,
        "test_plan": plan,
        "generated_tests": tests,
        "review": review
    }


print("Multi-agent workflow created successfully!")

Multi-agent workflow created successfully!


In [19]:
source_code = """
def calculate_total(price, quantity):
    if price < 0 or quantity < 0:
        raise ValueError("Inputs cannot be negative")
    return price * quantity
"""

result = run_multi_agent_system(source_code)

1. Code Analyzer Agent running...
2. Test Planner Agent running...
3. Test Generator Agent running...
4. Test Reviewer Agent running...

All four agents completed!


In [20]:
print("===== 1. CODE ANALYSIS =====")
print(result["analysis"])

print("\n===== 2. TEST PLAN =====")
print(result["test_plan"])

print("\n===== 3. GENERATED UNIT TESTS =====")
print(result["generated_tests"])

print("\n===== 4. REVIEWER FEEDBACK =====")
print(result["review"])

===== 1. CODE ANALYSIS =====
Here is the analysis of the provided Python source code:

### 1. What the code does
This code calculates the total cost of an item by multiplying its price by its quantity. It also includes a safety check to ensure that neither the price nor the quantity is a negative number. If a negative number is provided, it stops the program and reports an error.

---

### 2. The functions and parameters

* **Function Name:** `calculate_total`
* **Parameters:**
  * `price` (int or float): The cost of a single item.
  * `quantity` (int or float): The number of items being purchased.

---

### 3. The expected output
* **Normal Case:** Returns the product of `price` and `quantity` as an integer or float.
  * *Example:* `calculate_total(10, 3)` returns `30`
* **Error Case:** Raises a `ValueError` with the message `"Inputs cannot be negative"` if either parameter is less than zero.

---

### 4. Possible errors or edge cases

* **Negative Numbers:** Passing a negative price 

In [21]:
from pathlib import Path

Path("user_code.py").write_text(source_code)
Path("test_generated.py").write_text(
    result["generated_tests"]
)

print("Saved user_code.py")
print("Saved test_generated.py")

Saved user_code.py
Saved test_generated.py


In [22]:
import subprocess
import sys

run = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-v",
        "--tb=short",
        "test_generated.py"
    ],
    capture_output=True,
    text=True,
    timeout=30
)

print(run.stdout)

if run.stderr:
    print(run.stderr)

print("Pytest exit code:", run.returncode)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.1.1, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.4
collecting ... collected 7 items

test_generated.py::test_calculate_total_normal PASSED                    [ 14%]
test_generated.py::test_calculate_total_decimal PASSED                   [ 28%]
test_generated.py::test_calculate_total_zero_price PASSED                [ 42%]
test_generated.py::test_calculate_total_zero_quantity PASSED             [ 57%]
test_generated.py::test_calculate_total_negative_price PASSED            [ 71%]
test_generated.py::test_calculate_total_negative_quantity PASSED         [ 85%]
test_generated.py::test_calculate_total_non_numeric_type PASSED          [100%]

============================== 7 passed in 0.04s ===============================

Pytest exit code: 0


In [23]:
import re

output = run.stdout + "\n" + run.stderr

passed = sum(
    int(n) for n in re.findall(r"(\d+) passed", output)
)

failed = sum(
    int(n) for n in re.findall(r"(\d+) failed", output)
)

errors = sum(
    int(n) for n in re.findall(r"(\d+) error", output)
)

print("===== TEST EXECUTION SUMMARY =====")
print("Tests passed:", passed)
print("Tests failed:", failed)
print("Test errors:", errors)

total = passed + failed

if total:
    print("Pass rate:", round(passed / total * 100, 2), "%")
else:
    print("Pass rate unavailable: no test results were detected.")

===== TEST EXECUTION SUMMARY =====
Tests passed: 7
Tests failed: 0
Test errors: 0
Pass rate: 100.0 %


In [24]:
!pip -q install pytest-cov

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.0/258.0 kB 13.5 MB/s eta 0:00:00


In [25]:
import subprocess
import sys

coverage_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "--cov=user_code",
        "--cov-report=term-missing",
        "test_generated.py"
    ],
    capture_output=True,
    text=True,
    timeout=30
)

print(coverage_run.stdout)

if coverage_run.stderr:
    print(coverage_run.stderr)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.1.1, pluggy-1.7.0
rootdir: /content
plugins: cov-7.1.0, typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.4
collected 7 items

test_generated.py .......                                                [100%]

================================ tests coverage ================================
_______________ coverage: platform linux, python 3.13.16-final-0 _______________

Name           Stmts   Miss  Cover   Missing
--------------------------------------------
user_code.py       4      0   100%
--------------------------------------------
TOTAL              4      0   100%
============================== 7 passed in 0.20s ===============================



In [26]:
!pip -q install "mcp>=1.28,<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 4.4 MB/s eta 0:00:00


In [27]:
from mcp.server.fastmcp import FastMCP
import ast

mcp_server = FastMCP("Test Generation Tools")


@mcp_server.tool()
def analyze_source(source_code: str) -> str:
    """Check Python syntax and list function names."""

    try:
        tree = ast.parse(source_code)
    except SyntaxError as error:
        return f"Invalid Python syntax: {error}"

    functions = [
        node.name
        for node in ast.walk(tree)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    ]

    return (
        "Python syntax is valid.\n"
        f"Functions found: {functions}"
    )


print("MCP tool created successfully!")

MCP tool created successfully!


In [29]:
print(
    analyze_source("""
def calculate_total(price, quantity):
    return price * quantity
""")
)

Python syntax is valid.
Functions found: ['calculate_total']


In [30]:
%%writefile mcp_server.py
import ast
from mcp.server.fastmcp import FastMCP

mcp_server = FastMCP("Test Generation Tools")

@mcp_server.tool()
def analyze_source(source_code: str) -> str:
    """Check Python syntax and list function names."""
    try:
        tree = ast.parse(source_code)
    except SyntaxError as error:
        return f"Invalid Python syntax: {error}"

    functions = [
        node.name
        for node in ast.walk(tree)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    ]

    return (
        "Python syntax is valid.\n"
        f"Functions found: {functions}"
    )

if __name__ == "__main__":
    mcp_server.run(transport="stdio")

Writing mcp_server.py


In [33]:
import os
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command=sys.executable,
    args=["-u", "mcp_server.py"],
    env={**os.environ, "PYTHONUNBUFFERED": "1"}
)

async def test_mcp_connection():
    with open(os.devnull, "w") as error_log:
        async with stdio_client(
            server_params,
            errlog=error_log
        ) as (read_stream, write_stream):

            async with ClientSession(
                read_stream, write_stream
            ) as session:
                await session.initialize()

                tools = await session.list_tools()
                print("MCP connection successful!")
                print("Available tools:")

                for tool in tools.tools:
                    print("-", tool.name)

                result = await session.call_tool(
                    "analyze_source",
                    {"source_code": "def add(a, b):\n    return a + b"}
                )

                print("\nTool result:")
                for item in result.content:
                    if hasattr(item, "text"):
                        print(item.text)

await test_mcp_connection()

MCP connection successful!
Available tools:
- analyze_source

Tool result:
Python syntax is valid.
Functions found: ['add']


In [34]:
async def run_multi_agent_system_with_mcp(source_code):

    # Step 1: Validate source code through MCP
    print("1. MCP: Checking source code...")

    async with stdio_client(
        server_params,
        errlog=open(os.devnull, "w")
    ) as (read_stream, write_stream):

        async with ClientSession(
            read_stream, write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "analyze_source",
                {"source_code": source_code}
            )

            mcp_analysis = "\n".join(
                item.text
                for item in result.content
                if hasattr(item, "text")
            )

    print(mcp_analysis)

    # Stop if MCP reports invalid Python syntax
    if "Invalid Python syntax:" in mcp_analysis:
        return {"error": mcp_analysis}

    # Step 2: AI agents work together
    print("\n2. Code Analyzer Agent...")
    analysis = code_analyzer(source_code)

    print("3. Test Planner Agent...")
    plan = test_planner(source_code, analysis)

    print("4. Test Generator Agent...")
    tests = test_generator(source_code, plan)

    print("5. Test Reviewer Agent...")
    review = test_reviewer(source_code, tests)

    return {
        "mcp_analysis": mcp_analysis,
        "analysis": analysis,
        "test_plan": plan,
        "generated_tests": tests,
        "review": review
    }


print("MCP-integrated workflow created!")

MCP-integrated workflow created!


In [35]:
source_code = """
def calculate_total(price, quantity):
    if price < 0 or quantity < 0:
        raise ValueError("Inputs cannot be negative")
    return price * quantity
"""

result_mcp = await run_multi_agent_system_with_mcp(source_code)

1. MCP: Checking source code...
Python syntax is valid.
Functions found: ['calculate_total']

2. Code Analyzer Agent...
3. Test Planner Agent...
4. Test Generator Agent...
5. Test Reviewer Agent...


In [36]:
print("===== MCP SOURCE VALIDATION =====")
print(result_mcp["mcp_analysis"])

print("\n===== CODE ANALYSIS =====")
print(result_mcp["analysis"])

print("\n===== TEST PLAN =====")
print(result_mcp["test_plan"])

print("\n===== GENERATED UNIT TESTS =====")
print(result_mcp["generated_tests"])

print("\n===== TEST REVIEW =====")
print(result_mcp["review"])

===== MCP SOURCE VALIDATION =====
Python syntax is valid.
Functions found: ['calculate_total']

===== CODE ANALYSIS =====
Here is the analysis of the provided Python source code, broken down into the requested sections:

---

### 1. What the Code Does
This code calculates the total cost of an item by multiplying its price by the quantity purchased. It also includes a basic safety check to ensure that neither the price nor the quantity is a negative number. If a negative number is provided, it stops the program and reports an error.

---

### 2. The Functions and Parameters

*   **Function Name:** `calculate_total`
*   **Parameters:**
    *   `price` (number): The cost of a single unit of the item (can be an integer or a float, like `10` or `9.99`).
    *   `quantity` (number): The number of units being purchased (can be an integer or a float).

---

### 3. The Expected Output
*   **Normal Case:** If you pass valid positive numbers, it returns their product.
    *   *Example:* `calculat

In [37]:
%%writefile mcp_server.py
import ast
import os
from pathlib import Path
from mcp.server.fastmcp import FastMCP

mcp_server = FastMCP("Test Generation Tools")
WORKSPACE = Path("/content/multi_agent_workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)


@mcp_server.tool()
def analyze_source(source_code: str) -> str:
    """Check Python syntax and list function names."""
    try:
        tree = ast.parse(source_code)
    except SyntaxError as error:
        return f"Invalid Python syntax: {error}"

    functions = [
        node.name
        for node in ast.walk(tree)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    ]

    return (
        "Python syntax is valid.\n"
        f"Functions found: {functions}"
    )


@mcp_server.tool()
def save_generated_tests(test_code: str) -> str:
    """Validate Python syntax and save generated pytest code."""
    try:
        ast.parse(test_code)
    except SyntaxError as error:
        return f"Invalid test syntax: {error}"

    path = WORKSPACE / "test_generated.py"
    path.write_text(test_code, encoding="utf-8")
    return f"Tests saved successfully to {path}"


if __name__ == "__main__":
    mcp_server.run(transport="stdio")

Overwriting mcp_server.py


In [38]:
async def save_tests_with_mcp(test_code):
    with open(os.devnull, "w") as error_log:
        async with stdio_client(
            server_params,
            errlog=error_log
        ) as (read_stream, write_stream):

            async with ClientSession(
                read_stream, write_stream
            ) as session:

                await session.initialize()

                tools = await session.list_tools()
                print("Available MCP tools:",
                      [tool.name for tool in tools.tools])

                result = await session.call_tool(
                    "save_generated_tests",
                    {"test_code": test_code}
                )

                for item in result.content:
                    if hasattr(item, "text"):
                        print(item.text)


await save_tests_with_mcp(result_mcp["generated_tests"])

Available MCP tools: ['analyze_source', 'save_generated_tests']
Tests saved successfully to /content/multi_agent_workspace/test_generated.py


In [39]:
from pathlib import Path
import subprocess
import sys

workspace = Path("/content/multi_agent_workspace")

# Save the original source code beside the generated tests
(workspace / "user_code.py").write_text(
    source_code,
    encoding="utf-8"
)

# Execute the tests
test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-v",
        "--tb=short",
        "test_generated.py"
    ],
    cwd=str(workspace),
    capture_output=True,
    text=True,
    timeout=30
)

print(test_run.stdout)

if test_run.stderr:
    print(test_run.stderr)

print("Exit code:", test_run.returncode)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.1.1, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/multi_agent_workspace
plugins: cov-7.1.0, typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.4
collecting ... collected 9 items

test_generated.py::test_calculate_total_normal_integer PASSED            [ 11%]
test_generated.py::test_calculate_total_normal_float PASSED              [ 22%]
test_generated.py::test_calculate_total_zero_price PASSED                [ 33%]
test_generated.py::test_calculate_total_zero_quantity PASSED             [ 44%]
test_generated.py::test_calculate_total_both_zero PASSED                 [ 55%]
test_generated.py::test_calculate_total_negative_price PASSED            [ 66%]
test_generated.py::test_calculate_total_negative_quantity PASSED         [ 77%]
test_generated.py::test_calculate_total_both_negative PASSED             [ 88%]
test_gener

In [40]:
%%writefile mcp_server.py
import ast
import json
from pathlib import Path
from mcp.server.fastmcp import FastMCP

mcp_server = FastMCP("Test Generation Tools")
WORKSPACE = Path("/content/multi_agent_workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)


@mcp_server.tool()
def analyze_source(source_code: str) -> str:
    """Check Python syntax and list function names."""
    try:
        tree = ast.parse(source_code)
    except SyntaxError as error:
        return f"Invalid Python syntax: {error}"

    functions = [
        node.name for node in ast.walk(tree)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    ]
    return f"Python syntax is valid.\nFunctions found: {functions}"


@mcp_server.tool()
def save_generated_tests(test_code: str) -> str:
    """Validate syntax and save generated pytest code."""
    try:
        ast.parse(test_code)
    except SyntaxError as error:
        return f"Invalid test syntax: {error}"

    path = WORKSPACE / "test_generated.py"
    path.write_text(test_code, encoding="utf-8")
    return f"Tests saved successfully to {path}"


@mcp_server.tool()
def record_evaluation(report_json: str) -> str:
    """Validate and save an evaluation report as JSON."""
    try:
        report = json.loads(report_json)
    except json.JSONDecodeError as error:
        return f"Invalid JSON report: {error}"

    path = WORKSPACE / "evaluation_report.json"
    path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return f"Evaluation report saved to {path}"


if __name__ == "__main__":
    mcp_server.run(transport="stdio")

Overwriting mcp_server.py


In [41]:
import json

evaluation_summary = {
    "system": "Multi-Agent Test Generation with Gemini and MCP",
    "sample_function": "calculate_total",
    "agents": 4,
    "mcp_tools": [
        "analyze_source",
        "save_generated_tests",
        "record_evaluation"
    ],
    "tests_executed": 9,
    "tests_passed": 9,
    "tests_failed": 0,
    "pass_rate_percent": 100.0,
    "execution_time_seconds": 0.03
}

await save_tests_with_mcp(result_mcp["generated_tests"])  # Optional: save tests again

async def record_report_with_mcp(report):
    with open(os.devnull, "w") as error_log:
        async with stdio_client(
            server_params,
            errlog=error_log
        ) as (read_stream, write_stream):
            async with ClientSession(read_stream, write_stream) as session:
                await session.initialize()

                result = await session.call_tool(
                    "record_evaluation",
                    {"report_json": json.dumps(report)}
                )

                for item in result.content:
                    if hasattr(item, "text"):
                        print(item.text)

await record_report_with_mcp(evaluation_summary)

Available MCP tools: ['analyze_source', 'save_generated_tests', 'record_evaluation']
Tests saved successfully to /content/multi_agent_workspace/test_generated.py
Evaluation report saved to /content/multi_agent_workspace/evaluation_report.json


In [42]:
import json
from pathlib import Path

report_path = Path(
    "/content/multi_agent_workspace/evaluation_report.json"
)

report = json.loads(report_path.read_text())

print("===== SAVED EVALUATION REPORT =====")
print(json.dumps(report, indent=2))

===== SAVED EVALUATION REPORT =====
{
  "system": "Multi-Agent Test Generation with Gemini and MCP",
  "sample_function": "calculate_total",
  "agents": 4,
  "mcp_tools": [
    "analyze_source",
    "save_generated_tests",
    "record_evaluation"
  ],
  "tests_executed": 9,
  "tests_passed": 9,
  "tests_failed": 0,
  "pass_rate_percent": 100.0,
  "execution_time_seconds": 0.03
}


In [43]:
evaluation_cases = [
    {
        "id": "E01",
        "name": "Addition",
        "source": """
def add(a, b):
    return a + b
"""
    },
    {
        "id": "E02",
        "name": "String processing",
        "source": """
def normalize_name(name):
    if not isinstance(name, str):
        raise TypeError("Name must be a string")
    return name.strip().title()
"""
    },
    {
        "id": "E03",
        "name": "Input validation",
        "source": """
def calculate_total(price, quantity):
    if price < 0 or quantity < 0:
        raise ValueError("Inputs cannot be negative")
    return price * quantity
"""
    },
    {
        "id": "E04",
        "name": "Invalid Python syntax",
        "source": """
def broken_function(
    return 10
"""
    }
]

print("Evaluation cases prepared:", len(evaluation_cases))

for case in evaluation_cases:
    print(case["id"], "-", case["name"])

Evaluation cases prepared: 4
E01 - Addition
E02 - String processing
E03 - Input validation
E04 - Invalid Python syntax


In [44]:
import ast
import subprocess
import sys
from pathlib import Path

case = evaluation_cases[0]

print("Evaluating:", case["name"])

# 1. Run MCP + all four AI agents
result = await run_multi_agent_system_with_mcp(case["source"])

# 2. Stop if the workflow returned an error
if "error" in result:
    print("Workflow error:", result["error"])

else:
    # 3. Save source code and generated tests
    workspace = Path("/content/evaluation_E01")
    workspace.mkdir(parents=True, exist_ok=True)

    (workspace / "user_code.py").write_text(
        case["source"], encoding="utf-8"
    )

    test_code = result["generated_tests"]

    # Remove Markdown fences if Gemini included them
    if test_code.startswith("```"):
        lines = test_code.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        test_code = "\n".join(lines)

    # 4. Check that generated tests are valid Python
    ast.parse(test_code)

    (workspace / "test_generated.py").write_text(
        test_code, encoding="utf-8"
    )

    # 5. Execute the tests
    test_run = subprocess.run(
        [
            sys.executable, "-m", "pytest",
            "-v", "--tb=short", "test_generated.py"
        ],
        cwd=str(workspace),
        capture_output=True,
        text=True,
        timeout=30
    )

    print("\n===== TEST RESULTS =====")
    print(test_run.stdout)

    if test_run.stderr:
        print(test_run.stderr)

    print("Pytest exit code:", test_run.returncode)

Evaluating: Addition
1. MCP: Checking source code...
Python syntax is valid.
Functions found: ['add']

2. Code Analyzer Agent...
3. Test Planner Agent...
4. Test Generator Agent...
5. Test Reviewer Agent...

===== TEST RESULTS =====
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.1.1, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/evaluation_E01
plugins: cov-7.1.0, typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.4
collecting ... collected 12 items

test_generated.py::test_add_positive_integers PASSED                     [  8%]
test_generated.py::test_add_floating_point_numbers PASSED                [ 16%]
test_generated.py::test_add_strings PASSED                               [ 25%]
test_generated.py::test_add_lists PASSED                                 [ 33%]
test_generated.py::test_add_zero_values PASSED                           [ 41%]
test_generated.py::test

In [45]:
case = evaluation_cases[1]

print("Evaluating:", case["name"])

result_E02 = await run_multi_agent_system_with_mcp(
    case["source"]
)

if "error" in result_E02:
    print("Workflow error:", result_E02["error"])

else:
    workspace = Path("/content/evaluation_E02")
    workspace.mkdir(parents=True, exist_ok=True)

    (workspace / "user_code.py").write_text(
        case["source"], encoding="utf-8"
    )

    test_code = result_E02["generated_tests"].strip()

    # Remove optional Markdown code fences
    if test_code.startswith("```"):
        lines = test_code.splitlines()
        lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        test_code = "\n".join(lines)

    try:
        ast.parse(test_code)

        (workspace / "test_generated.py").write_text(
            test_code, encoding="utf-8"
        )

        test_run_E02 = subprocess.run(
            [
                sys.executable, "-m", "pytest",
                "-v", "--tb=short", "test_generated.py"
            ],
            cwd=str(workspace),
            capture_output=True,
            text=True,
            timeout=30
        )

        print("\n===== E02 TEST RESULTS =====")
        print(test_run_E02.stdout)

        if test_run_E02.stderr:
            print(test_run_E02.stderr)

        print("Pytest exit code:", test_run_E02.returncode)

    except SyntaxError as exc:
        print("Generated tests have invalid Python syntax:", exc)

Evaluating: String processing
1. MCP: Checking source code...
Python syntax is valid.
Functions found: ['normalize_name']

2. Code Analyzer Agent...
3. Test Planner Agent...
4. Test Generator Agent...
5. Test Reviewer Agent...

===== E02 TEST RESULTS =====
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.1.1, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/evaluation_E02
plugins: cov-7.1.0, typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.4
collecting ... collected 3 items

test_generated.py::test_normalize_name_normal_cases PASSED               [ 33%]
test_generated.py::test_normalize_name_boundary_cases PASSED             [ 66%]
test_generated.py::test_normalize_name_invalid_inputs PASSED             [100%]

============================== 3 passed in 0.02s ===============================

Pytest exit code: 0


In [46]:
case = evaluation_cases[2]

print("Evaluating:", case["name"])

result_E03 = await run_multi_agent_system_with_mcp(
    case["source"]
)

if "error" in result_E03:
    print("Workflow error:", result_E03["error"])

else:
    workspace = Path("/content/evaluation_E03")
    workspace.mkdir(parents=True, exist_ok=True)

    (workspace / "user_code.py").write_text(
        case["source"], encoding="utf-8"
    )

    test_code = result_E03["generated_tests"].strip()

    # Remove optional Markdown code fences
    if test_code.startswith("```"):
        lines = test_code.splitlines()[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        test_code = "\n".join(lines)

    try:
        ast.parse(test_code)

        (workspace / "test_generated.py").write_text(
            test_code, encoding="utf-8"
        )

        test_run_E03 = subprocess.run(
            [
                sys.executable, "-m", "pytest",
                "-v", "--tb=short", "test_generated.py"
            ],
            cwd=str(workspace),
            capture_output=True,
            text=True,
            timeout=30
        )

        print("\n===== E03 TEST RESULTS =====")
        print(test_run_E03.stdout)

        if test_run_E03.stderr:
            print(test_run_E03.stderr)

        print("Pytest exit code:", test_run_E03.returncode)

    except SyntaxError as exc:
        print("Generated tests have invalid Python syntax:", exc)

Evaluating: Input validation
1. MCP: Checking source code...
Python syntax is valid.
Functions found: ['calculate_total']

2. Code Analyzer Agent...
3. Test Planner Agent...
4. Test Generator Agent...
5. Test Reviewer Agent...

===== E03 TEST RESULTS =====
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.1.1, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/evaluation_E03
plugins: cov-7.1.0, typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.4
collecting ... collected 8 items

test_generated.py::test_calculate_total_positive_integers PASSED         [ 12%]
test_generated.py::test_calculate_total_positive_floats PASSED           [ 25%]
test_generated.py::test_calculate_total_zero_price PASSED                [ 37%]
test_generated.py::test_calculate_total_zero_quantity PASSED             [ 50%]
test_generated.py::test_calculate_total_zero_both PASSED                 [ 62%]


In [47]:
case = evaluation_cases[3]

print("Evaluating:", case["name"])

with open(os.devnull, "w") as error_log:
    async with stdio_client(
        server_params,
        errlog=error_log
    ) as (read_stream, write_stream):

        async with ClientSession(
            read_stream, write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "analyze_source",
                {"source_code": case["source"]}
            )

            for item in result.content:
                if hasattr(item, "text"):
                    print(item.text)

Evaluating: Invalid Python syntax
Invalid Python syntax: '(' was never closed (<unknown>, line 2)


In [48]:
async def orchestrate(source_code):
    result = {
        "status": "started",
        "mcp_validation": None,
        "analysis": None,
        "test_plan": None,
        "generated_tests": None,
        "review": None,
        "error": None
    }

    try:
        # 1. Validate using MCP
        print("1. MCP validation...")

        with open(os.devnull, "w") as error_log:
            async with stdio_client(
                server_params,
                errlog=error_log
            ) as (read_stream, write_stream):

                async with ClientSession(
                    read_stream, write_stream
                ) as session:
                    await session.initialize()

                    tool_result = await session.call_tool(
                        "analyze_source",
                        {"source_code": source_code}
                    )

                    mcp_output = "\n".join(
                        item.text
                        for item in tool_result.content
                        if hasattr(item, "text")
                    )

        result["mcp_validation"] = mcp_output

        if "Invalid Python syntax:" in mcp_output:
            result["status"] = "rejected"
            result["error"] = mcp_output
            print(mcp_output)
            return result

        # 2. Run the agents in sequence
        print("2. Code Analyzer...")
        result["analysis"] = code_analyzer(source_code)

        print("3. Test Planner...")
        result["test_plan"] = test_planner(
            source_code, result["analysis"]
        )

        print("4. Test Generator...")
        result["generated_tests"] = test_generator(
            source_code, result["test_plan"]
        )

        print("5. Test Reviewer...")
        result["review"] = test_reviewer(
            source_code, result["generated_tests"]
        )

        result["status"] = "completed"
        print("Workflow completed successfully!")

    except Exception as exc:
        result["status"] = "failed"
        result["error"] = f"{type(exc).__name__}: {exc}"
        print("Workflow failed:", result["error"])

    return result


print("Improved orchestrator created successfully!")

Improved orchestrator created successfully!


In [49]:
demo = await orchestrate("""
def multiply(a, b):
    return a * b
""")

print("\nFinal status:", demo["status"])
print("\nGenerated tests:\n", demo["generated_tests"])

1. MCP validation...
2. Code Analyzer...
3. Test Planner...
4. Test Generator...
5. Test Reviewer...
Workflow completed successfully!

Final status: completed

Generated tests:
 import pytest
from user_code import multiply


def test_multiply_positive_integers():
    assert multiply(3, 4) == 12


def test_multiply_positive_and_negative_integer():
    assert multiply(5, -2) == -10


def test_multiply_floating_point_numbers():
    assert multiply(2.5, 4.0) == pytest.approx(10.0)


def test_multiply_by_zero():
    assert multiply(0, 5) == 0


def test_multiply_by_one():
    assert multiply(1, 99) == 99


def test_multiply_large_numbers():
    assert multiply(1e308, 1) == pytest.approx(1e308)


def test_multiply_invalid_types():
    with pytest.raises(TypeError):
        multiply("hello", "world")


def test_multiply_missing_argument():
    with pytest.raises(TypeError):
        multiply(5)


def test_multiply_too_many_arguments():
    with pytest.raises(TypeError):
        multiply(1, 2, 

In [50]:
invalid_demo = await orchestrate("""
def broken(
    return 5
""")

print("\nInvalid source status:", invalid_demo["status"])

1. MCP validation...
Invalid Python syntax: '(' was never closed (<unknown>, line 2)

Invalid source status: rejected


In [51]:
orchestration_summary = {
    "project": "Multi-Agent Test Generation System",
    "orchestration": "Sequential",
    "mcp_validation": True,
    "agents": [
        "Code Analyzer",
        "Test Planner",
        "Test Generator",
        "Test Reviewer"
    ],
    "valid_source_status": demo["status"],
    "invalid_source_status": invalid_demo["status"],
    "valid_source_handled": demo["status"] == "completed",
    "invalid_source_handled": invalid_demo["status"] == "rejected"
}

await record_report_with_mcp(orchestration_summary)

Evaluation report saved to /content/multi_agent_workspace/evaluation_report.json


In [52]:
orchestration_summary = {
    "project": "Multi-Agent Test Generation System",
    "orchestration": "Sequential",
    "mcp_validation": True,
    "agents": [
        "Code Analyzer",
        "Test Planner",
        "Test Generator",
        "Test Reviewer"
    ],
    "valid_source_status": demo["status"],
    "invalid_source_status": invalid_demo["status"],
    "valid_source_handled": demo["status"] == "completed",
    "invalid_source_handled": invalid_demo["status"] == "rejected"
}

await record_report_with_mcp(orchestration_summary)

Evaluation report saved to /content/multi_agent_workspace/evaluation_report.json


In [53]:

import os
import sys
import json
import subprocess

evaluation_folders = {
    "E01 - Addition": "/content/evaluation_E01",
    "E02 - String processing": "/content/evaluation_E02",
    "E03 - Input validation": "/content/evaluation_E03"
}

coverage_results = []

for case_name, folder in evaluation_folders.items():
    print(f"\n{'=' * 50}")
    print(f"Evaluating: {case_name}")
    print("=" * 50)

    command = [
        sys.executable, "-m", "pytest",
        "--cov=user_code",
        "--cov-report=term-missing",
        "--cov-report=json:coverage.json",
        "test_generated.py"
    ]

    result = subprocess.run(
        command,
        cwd=folder,
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.stderr:
        print("Warnings or errors:")
        print(result.stderr)

    coverage_file = os.path.join(folder, "coverage.json")

    if os.path.exists(coverage_file):
        with open(coverage_file, "r", encoding="utf-8") as file:
            coverage_data = json.load(file)

        totals = coverage_data["totals"]

        coverage_results.append({
            "case": case_name,
            "coverage_percent": totals["percent_covered"],
            "lines_covered": totals["covered_lines"],
            "total_lines": totals["num_statements"],
            "pytest_exit_code": result.returncode
        })
    else:
        coverage_results.append({
            "case": case_name,
            "coverage_percent": None,
            "lines_covered": None,
            "total_lines": None,
            "pytest_exit_code": result.returncode
        })

print("\nCoverage evaluation finished.")
print(json.dumps(coverage_results, indent=2))



Evaluating: E01 - Addition
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-9.1.1, pluggy-1.7.0
rootdir: /content/evaluation_E01
plugins: cov-7.1.0, typeguard-4.6.0, anyio-4.15.1, langsmith-0.14.11, platformdirs-4.13.4
collected 12 items

test_generated.py ............                                           [100%]

================================ tests coverage ================================
_______________ coverage: platform linux, python 3.13.16-final-0 _______________

Name           Stmts   Miss  Cover   Missing
--------------------------------------------
user_code.py       2      0   100%
--------------------------------------------
TOTAL              2      0   100%
Coverage JSON written to file coverage.json
============================== 12 passed in 0.08s ==============================


Evaluating: E02 - String processing
============================= test session starts =========================

In [54]:

evaluation_summary = {
    "project": "Multi-Agent Test Generation System",
    "evaluation_cases": [
        {
            "id": "E01",
            "name": "Addition",
            "tests_passed": 12,
            "tests_failed": 0,
            "statement_coverage_percent": 100.0,
            "execution_time_seconds": 0.08
        },
        {
            "id": "E02",
            "name": "String processing",
            "tests_passed": 3,
            "tests_failed": 0,
            "statement_coverage_percent": 100.0,
            "execution_time_seconds": 0.06
        },
        {
            "id": "E03",
            "name": "Input validation",
            "tests_passed": 8,
            "tests_failed": 0,
            "statement_coverage_percent": 100.0,
            "execution_time_seconds": 0.07
        }
    ],
    "total_tests_passed": 23,
    "total_tests_failed": 0,
    "invalid_syntax_case": {
        "id": "E04",
        "expected_behavior": "Reject invalid Python syntax",
        "result": "Rejected correctly"
    },
    "evaluation_note": (
        "Coverage and test results apply to these three valid examples; "
        "they do not establish general reliability."
    )
}

await record_report_with_mcp(evaluation_summary)


Evaluation report saved to /content/multi_agent_workspace/evaluation_report.json


In [55]:

%pip install -q gradio


In [56]:

import gradio as gr

print("Gradio version:", gr.__version__)
print("Gradio is ready!")


Gradio version: 6.30.0
Gradio is ready!


In [57]:

import gradio as gr
import asyncio

def generate_tests_interface(source_code):
    if not source_code or not source_code.strip():
        return (
            "Please enter Python source code.",
            "",
            "",
            "",
            ""
        )

    try:
        result = asyncio.run(orchestrate(source_code))

        if result["status"] != "completed":
            error_message = (
                f"Status: {result['status']}\n"
                f"Error: {result.get('error') or 'Workflow did not complete.'}"
            )
            return error_message, "", "", "", ""

        return (
            result.get("mcp_validation", ""),
            result.get("analysis", ""),
            result.get("test_plan", ""),
            result.get("generated_tests", ""),
            result.get("review", "")
        )

    except Exception as error:
        return (
            f"Error: {type(error).__name__}: {error}",
            "",
            "",
            "",
            ""
        )


with gr.Blocks(title="Multi-Agent Test Generation Agent") as app:
    gr.Markdown("# Multi-Agent Test Generation Agent")
    gr.Markdown(
        "Generate and review Python unit tests using four AI agents "
        "and MCP validation."
    )

    source_input = gr.Code(
        label="Enter Python source code",
        language="python",
        value="def add(a, b):\n    return a + b",
        lines=8
    )

    generate_button = gr.Button("Generate Tests", variant="primary")

    validation_output = gr.Textbox(label="MCP Validation", lines=3)
    analysis_output = gr.Textbox(label="1. Code Analyzer Agent", lines=5)
    plan_output = gr.Textbox(label="2. Test Planner Agent", lines=7)
    tests_output = gr.Code(label="3. Test Generator Agent", language="python")
    review_output = gr.Textbox(label="4. Test Reviewer Agent", lines=6)

    generate_button.click(
        fn=generate_tests_interface,
        inputs=[source_input],
        outputs=[
            validation_output,
            analysis_output,
            plan_output,
            tests_output,
            review_output
        ]
    )

app.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1cdd19e84983b8adf9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [58]:

import gradio as gr
import asyncio
import os
import tempfile

def generate_tests_interface(source_code):
    if not source_code or not source_code.strip():
        return "Please enter Python source code.", "", "", "", "", None

    try:
        result = asyncio.run(orchestrate(source_code))

        if result["status"] != "completed":
            error = (
                f"Status: {result['status']}\n"
                f"Error: {result.get('error') or 'Workflow did not complete.'}"
            )
            return error, "", "", "", "", None

        test_code = result.get("generated_tests", "")

        download_path = None
        if test_code.strip():
            fd, download_path = tempfile.mkstemp(
                suffix="_test_generated.py"
            )
            with os.fdopen(fd, "w", encoding="utf-8") as file:
                file.write(test_code)

        return (
            result.get("mcp_validation", ""),
            result.get("analysis", ""),
            result.get("test_plan", ""),
            test_code,
            result.get("review", ""),
            download_path
        )

    except Exception as error:
        return (
            f"Error: {type(error).__name__}: {error}",
            "", "", "", "", None
        )


with gr.Blocks(title="Multi-Agent Test Generation Agent") as app:
    gr.Markdown("# Multi-Agent Test Generation Agent")
    gr.Markdown(
        "Generate and review Python unit tests using four AI agents "
        "and MCP validation."
    )

    source_input = gr.Code(
        label="Python source code",
        language="python",
        value="def add(a, b):\n    return a + b",
        lines=8
    )

    generate_button = gr.Button("Generate Tests", variant="primary")

    validation_output = gr.Textbox(label="MCP Validation", lines=3)
    analysis_output = gr.Textbox(label="1. Code Analyzer Agent", lines=5)
    plan_output = gr.Textbox(label="2. Test Planner Agent", lines=7)
    tests_output = gr.Code(label="3. Test Generator Agent", language="python")
    review_output = gr.Textbox(label="4. Test Reviewer Agent", lines=6)

    download_output = gr.File(label="Download generated pytest file")

    generate_button.click(
        fn=generate_tests_interface,
        inputs=[source_input],
        outputs=[
            validation_output,
            analysis_output,
            plan_output,
            tests_output,
            review_output,
            download_output
        ]
    )

app.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a96a8d94cd78113709.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [59]:

import os
import zipfile
import tempfile

def create_project_zip(source_code, test_code):
    if not source_code.strip() or not test_code.strip():
        return None

    temp_dir = tempfile.mkdtemp()
    zip_path = os.path.join(temp_dir, "test_generation_project.zip")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zip_file:
        zip_file.writestr("user_code.py", source_code)
        zip_file.writestr("test_generated.py", test_code)
        zip_file.writestr(
            "README.txt",
            "Multi-Agent Test Generation Project\n\n"
            "Files:\n"
            "- user_code.py: Your original Python source code\n"
            "- test_generated.py: AI-generated pytest tests\n\n"
            "To run tests locally:\n"
            "1. Install pytest: pip install pytest\n"
            "2. Run: pytest test_generated.py\n\n"
            "Review generated tests before running them."
        )

    return zip_path

print("ZIP creation function is ready!")


ZIP creation function is ready!


In [60]:
project_zip_output = gr.File(label="Download complete project ZIP")

In [61]:

import gradio as gr
import asyncio
import os
import tempfile
import zipfile

def generate_tests_interface(source_code):
    if not source_code or not source_code.strip():
        return ("Please enter Python source code.", "", "", "", "", None, None)

    try:
        result = asyncio.run(orchestrate(source_code))

        if result["status"] != "completed":
            error = (
                f"Status: {result['status']}\n"
                f"Error: {result.get('error') or 'Workflow did not complete.'}"
            )
            return error, "", "", "", "", None, None

        test_code = result.get("generated_tests", "")
        individual_path = None
        zip_path = None

        if test_code.strip():
            temp_dir = tempfile.mkdtemp()

            individual_path = os.path.join(
                temp_dir, "test_generated.py"
            )
            with open(individual_path, "w", encoding="utf-8") as file:
                file.write(test_code)

            zip_path = os.path.join(
                temp_dir, "test_generation_project.zip"
            )

            with zipfile.ZipFile(
                zip_path, "w", zipfile.ZIP_DEFLATED
            ) as zip_file:
                zip_file.writestr("user_code.py", source_code)
                zip_file.writestr("test_generated.py", test_code)
                zip_file.writestr(
                    "README.txt",
                    "Multi-Agent Test Generation Project\n\n"
                    "Files:\n"
                    "- user_code.py: Original Python source code\n"
                    "- test_generated.py: AI-generated pytest tests\n\n"
                    "To run tests locally:\n"
                    "1. Install pytest: pip install pytest\n"
                    "2. Run: pytest test_generated.py\n\n"
                    "Review generated tests before running them."
                )

        return (
            result.get("mcp_validation", ""),
            result.get("analysis", ""),
            result.get("test_plan", ""),
            test_code,
            result.get("review", ""),
            individual_path,
            zip_path
        )

    except Exception as error:
        return (
            f"Error: {type(error).__name__}: {error}",
            "", "", "", "", None, None
        )


with gr.Blocks(title="Multi-Agent Test Generation Agent") as app:
    gr.Markdown("# Multi-Agent Test Generation Agent")
    gr.Markdown(
        "Generate and review Python unit tests using four AI agents "
        "and MCP validation."
    )

    source_input = gr.Code(
        label="Python source code",
        language="python",
        value="def add(a, b):\n    return a + b",
        lines=8
    )

    generate_button = gr.Button("Generate Tests", variant="primary")

    validation_output = gr.Textbox(label="MCP Validation", lines=3)
    analysis_output = gr.Textbox(label="1. Code Analyzer Agent", lines=5)
    plan_output = gr.Textbox(label="2. Test Planner Agent", lines=7)
    tests_output = gr.Code(
        label="3. Test Generator Agent",
        language="python"
    )
    review_output = gr.Textbox(label="4. Test Reviewer Agent", lines=6)

    download_output = gr.File(label="Download generated test file")
    project_zip_output = gr.File(label="Download complete project ZIP")

    generate_button.click(
        fn=generate_tests_interface,
        inputs=[source_input],
        outputs=[
            validation_output,
            analysis_output,
            plan_output,
            tests_output,
            review_output,
            download_output,
            project_zip_output
        ]
    )

app.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4e7859256a33554129.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [62]:

requirements = """gradio
google-genai
mcp
pytest
pytest-cov
"""

with open("/content/requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully!")
print(requirements)


requirements.txt created successfully!
gradio
google-genai
mcp
pytest
pytest-cov



In [63]:

from google.colab import files

files.download("/content/requirements.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>